In [ ]:
import os
import re

import matplotlib.pyplot as plt
import pandas as pd

from glob import glob

from skimage import measure
from tqdm import tqdm

import seaborn as sns

In [186]:

# srs_df_path = r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium WSI\masking_outputs\summary_masked_instances.csv"
srs_df_path = r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium HSI\masking_outputs\summary_masked_instances.csv"

# xenium_parquet_files = [
#     r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium_data\Xen24_3782_aki-selected\aki_cell_segmentation-roi1_cell_centroids.parquet",
#     r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium_data\Xen24_3782_aki-selected\aki_cell_segmentation-roi2_cell_centroids.parquet",
#     r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium_data\Xen24_3990_ckd-selected\ckd_cell_segmentation-roi1_cell_centroids.parquet",
#     r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium_data\Xen24_3990_ckd-selected\ckd_cell_segmentation-roi2_cell_centroids.parquet",
#     r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium_data\Xen24_KPMP_038_ref-selected\ref_cell_segmentation-roi1_cell_centroids.parquet",
#     r"D:\ADATA Backup\HuBMAP\HuBMAP Xenium\Xenium_data\Xen24_KPMP_038_ref-selected\ref_cell_segmentation-roi2_cell_centroids.parquet"
# ]
# file_dfs = []
# for file in xenium_parquet_files:

#     df = pd.read_parquet(file)
#     df['Source_ID'] = os.path.basename(file).split('_')[0]

#     file_dfs.append(df)
# xenium_df = pd.concat(file_dfs, axis=0)

# xenium_df['Source_ID'] = xenium_df['Source_ID'].astype(str)
srs_df = pd.read_csv(srs_df_path)
srs_df['Source_ID'] = srs_df['Source_ID'].astype(str)
srs_df['Source_ID'] = srs_df['Source_ID'].str.upper()


output_dir = os.path.join(os.path.dirname(srs_df_path), 'summary_data_outputs_mol')
os.makedirs(output_dir, exist_ok=True)

In [ ]:
srs_df.columns

In [187]:
data_cols = [
    'Unit',
    'Source_ID',
    'Instance_Label',

    # 'Lipid_to_Protein',
    # 'Lipid_Unsaturation',

    # 'Collagen_Thickness',
    # 'Collagen_Coherence'
    
]

value_cols = [col for col in srs_df.columns if 'CH' in col]
value_cols.append('Redox')
data_cols.extend(value_cols)
data_df = srs_df[data_cols]

# data_df.rename(columns={'collagen_Collagen_Thickness': 'Collagen_Thickness', 'collagen_Collagen_Coherence': 'Collagen_Coherence'}, inplace=True)
data_df['Source_ID'] = data_df['Source_ID'].str.upper()
data_df = data_df.loc[:, ~data_df.columns.duplicated()].copy()
data_df.fillna(0, inplace=True)
data_df.rename(columns={'Instance_Label':'cell_id'}, inplace=True)
map = {
    'CKD':'3990',
    'AKI':'3872',
    'Ref':'KPMP_038'
}
data_df['Sample_Id'] = data_df['Source_ID'].map(map)

ch_cols = [col for col in srs_df.columns if 'CH' in col]
row_min = data_df[ch_cols[0:15]].mean(axis=1)
data_df[ch_cols] = data_df[ch_cols].sub(row_min, axis=0)
# data_df.to_csv(os.path.join(output_dir, "hubmap_round1_cell_mean_metabolism_data.csv"), index=False)

C:\Users\jvill\AppData\Local\Temp\ipykernel_16596\3959379606.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_df['Source_ID'] = data_df['Source_ID'].str.upper()


In [188]:
d_cols = [
    'Unit',
    'Source_ID',
    'cell_id',
]
srs_df.rename(columns={'Instance_Label':'cell_id'}, inplace=True)
sam_cols = [col for col in srs_df.columns if 'SAM' in col]
d_cols.extend(sam_cols)

In [ ]:
data_df = data_df[['Sample_Id'] + [col for col in data_df.columns if col != 'Sample_Id']]
data_df.to_csv(os.path.join(output_dir, "hubmap_round1_cell_mean_metabolism_data.csv"), index=False)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
label = data_df.iloc[:,:4]
X = data_df[ch_cols]
X_scaled = StandardScaler().fit_transform(X)

n_components=5
pca = PCA(n_components=n_components)

pca_features = pca.fit_transform(X)
pca_df = pd.DataFrame(pca_features, columns=[f"PC{i+1}" for i in range(n_components)])
pca_df[label.columns] = data_df[label.columns]

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
import umap

# label = data_df.iloc[:,:3]
# features = data_df[value_cols]
# features = features.fillna(0)
# reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
# embedding = reducer.fit_transform(features)
# df_umap = pd.DataFrame(embedding, columns=['UMAP_1', 'UMAP_2'])
# df_umap[label.columns] = data_df[label.columns].values

df_umap.to_csv(os.path.join(output_dir, "hubmap_round1_cell_mean_metabolism__umap_embeddings.csv"), index=False)
# data_df.to_csv(os.path.join(output_dir, "hubmap_round1_cell_mean_metabolism__features.csv"), index=False)

In [ ]:
import umap

label = df_2.iloc[:,:3]
features = df_2[value_cols]
features = features.fillna(0)
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
embedding = reducer.fit_transform(features)
df_2_umap = pd.DataFrame(embedding, columns=['UMAP_1', 'UMAP_2'])
df_2_umap[label.columns] = df_2[label.columns].values

df_2_umap.to_csv(os.path.join(output_dir, "hubmap_round1_cell_mean_metabolism_sub_umap_embeddings.csv"), index=False)
# df_2.to_csv(os.path.join(output_dir, "hubmap_round1_cell_mean_metabolism__features.csv"), index=False)

In [ ]:
# df_umap.to_csv(os.path.join(output_dir, "summary_masked_instances_umap_embeddings.csv"), index=False)
# data_df.to_csv(os.path.join(output_dir, "summary_masked_instances_features.csv"), index=False)

In [ ]:
df_umap = pd.read_csv((os.path.join(output_dir, "summary_masked_instances_umap_embeddings.csv")))

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- 1. Prepare Data & Shuffle ---
df_umap = df_2_umap.sample(frac=1, random_state=42).copy()

if "s" not in locals():
  s = 1.0

# --- 2. Visualization & Highlight Configuration ---
hue_col = "Source_ID"
unique_sources = list(df_umap[hue_col].dropna().unique())

# Set to specific Source_IDs to highlight (e.g. ['Source_1']), or None / 'all' to show all
highlight_sources = None
highlight_title = "Source ID"

# Background (Off-white) & Foreground styling - matching lineage plot
bg_color = "#e5e5e5"
bg_alpha = 0.35
fg_alpha = 0.90

# --- 3. Color Palette Configuration ---
# Turbo palette reordered as specified
base_palette = sns.color_palette("turbo", len(unique_sources))

if len(unique_sources) == 3:
  pal_order = [2, 0, 1]
  reordered_palette = [base_palette[i] for i in pal_order]
else:
  reordered_palette = base_palette

source_color_map = {
    src: col for src, col in zip(unique_sources, reordered_palette)
}

# Resolve active highlight set
if highlight_sources is None or highlight_sources == "all":
  active_highlight_set = set(unique_sources)
else:
  active_highlight_set = set(highlight_sources)

# Build plot color map
plot_colors = {}
for src in unique_sources:
  if src in active_highlight_set:
    plot_colors[src] = source_color_map[src]
  else:
    plot_colors[src] = bg_color

df_umap["color"] = df_umap[hue_col].map(plot_colors).fillna(bg_color)

# --- 4. Plotting (Exact parameters matching lineage cell) ---
fig, ax = plt.subplots(figsize=(18, 14), dpi=150)

# A. Plot Background (Off-white) points first
bg_subset = df_umap[df_umap["color"] == bg_color]
if not bg_subset.empty:
  ax.scatter(
      bg_subset["UMAP_1"],
      bg_subset["UMAP_2"],
      c=bg_color,
      label="Other / Unselected",
      alpha=bg_alpha,
      s=s * 1.5,
      edgecolors="none",
      rasterized=True,
  )

# B. Plot Highlighted sources on top
sources_to_plot = [src for src in unique_sources if src in active_highlight_set]

for src in sources_to_plot:
  subset = df_umap[df_umap[hue_col] == src]
  ax.scatter(
      subset["UMAP_1"],
      subset["UMAP_2"],
      c=[plot_colors[src]],
      label=str(src),
      alpha=fg_alpha,
      s=s * 2.2,
      edgecolors="none",
      rasterized=True,
  )

# --- 5. Formatting ---
title_str = (
    f"UMAP: {highlight_title}" if highlight_title else "UMAP: by Source ID"
)
ax.set_title(title_str, fontsize=22, fontweight="bold", pad=15)
ax.set_xlabel("UMAP 1", fontsize=16, labelpad=10)
ax.set_ylabel("UMAP 2", fontsize=16, labelpad=10)
ax.tick_params(axis="both", which="major", labelsize=12)
ax.grid(False)

# --- 6. Legend (Exact match to lineage legend styling) ---
ncols = 1 if len(sources_to_plot) <= 8 else (
    2 if len(sources_to_plot) <= 16 else 3
)
ax.legend(
    title=highlight_title,
    title_fontsize=14,
    fontsize=11,
    markerscale=3.0,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    ncol=ncols,
    frameon=True,
    framealpha=0.9,
)

plt.tight_layout()

# --- 7. Save Options ---
if "output_dir" in locals():
  os.makedirs(output_dir, exist_ok=True)
  plt.savefig(
      os.path.join(output_dir, "UMAP_Source_hubmap_round1.png"),
      dpi=300,
      bbox_inches="tight",
      transparent=False,
  )
  plt.savefig(
      os.path.join(output_dir, "UMAP_Source_hubmap_round1_trans.png"),
      dpi=300,
      bbox_inches="tight",
      transparent=True,
  )

plt.show()


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- 1. Prepare Data ---
df_umap = df_2_umap.copy()

if "s" not in locals():
  s = 1.0

# --- 2. Highlight Configuration ---
# Options:
#  - A list of specific cell labels: e.g. ['PC', 'altPC', 'IC', 'altIC', 'POD', 'PEC']
#  - A list of compartment names: e.g. ['Collecting Duct'] or ['Glomerular / Podocyte / PEC', 'Loop of Henle']
#  - Set to None or 'all' to show all cells with their full canonical colors
# highlight_labels = ["Glomerular / Podocyte / PEC", "Proximal Tubule", "Loop of Henle", "Distal / Connecting Tubule", "Collecting Duct"]
highlight_labels = ["Endothelial", "Stromal / Interstitium", 'Immune']
highlight_title = (
    "Endothelial, Interstitial, and Immune"  # Custom title for plot/legend
)

# Color & Alpha settings for background (non-highlighted) cells
bg_color = (
    "#c7c7c7"  # Off-white / light-grey background (e.g., '#e8e8e8', '#dedede')
)
bg_alpha = 0.5  # Transparency for background points
fg_alpha = 0.90  # Transparency for highlighted points

# --- 3. Canonical Kidney Color Hierarchy ---
kidney_color_groups = {
    # ── Glomerular / Podocytes & PEC (Reds, Magentas & Pinks) ──
    "Glomerular / Podocyte / PEC": {
        "POD": "#d00000",  # Crimson Red
        "altPOD": "#e63946",  # Vibrant Red
        "PEC": "#ff4d6d",  # Coral Rose
        "altPEC": "#ff758f",  # Light Pink
        "MC": "#a4133c",  # Deep Wine
        "EC-GC": "#ff8fa3",  # Soft Salmon
    },
    # ── Proximal Tubule (Bright Canary Yellows & Gold) ──
    "Proximal Tubule": {
        "PT": "#ffd166",  # Canary Gold
        "altPT": "#f4a261",  # Golden Ochre
    },
    # ── Loop of Henle (Warm Olives, Sage & Leaf Greens) ──
    "Loop of Henle": {
        "DTL": "#556b2f",  # Dark Olive
        "altDTL": "#6b8e23",  # Olive Drab
        "ATL": "#8fbc8f",  # Soft Sage
        "altATL": "#a3b18a",  # Pale Jade
        "TAL": "#107e3e",  # Forest/Kelly Green
        "altTAL": "#2d9344",  # Leaf Green
    },
    # ── Distal & Connecting Tubules (Purples & Violets) ──
    "Distal / Connecting Tubule": {
        "DCT": "#9b5de5",  # Bright Violet
        "altDCT": "#c77dff",  # Light Lavender
        "MD": "#7209b7",  # Deep Violet
        "CNT": "#560bad",  # Royal Purple
        "altCNT": "#3f0071",  # Dark Plum
    },
    # ── Collecting Duct: PC vs IC (Sky Blue vs Midnight Navy) ──
    "Collecting Duct": {
        "PC": "#00b4d8",  # Vivid Sky Blue (Principal Cells)
        "altPC": "#90e0ef",  # Light Aqua Blue
        "IC": "#03045e",  # Deep Midnight Navy (Intercalated Cells)
        "altIC": "#1d3557",  # Dark Steel Blue
    },
    # ── Endothelial (Turquoise / Cyan) ──
    "Endothelial": {
        "EC": "#00f5d4",  # Turquoise
    },
    # ── Stromal / Fibroblasts / Pericytes (Slate Greys & Muted Taupes) ──
    "Stromal / Interstitium": {
        "FIB": "#8d99ae",  # Slate Grey
        "aFIB": "#6c757d",  # Medium Grey
        "pvFIB": "#adb5bd",  # Pale Slate
        "infFIB": "#b08968",  # Warm Taupe
        "MYOF": "#7f5539",  # Nut Brown
        "VSM/P": "#583101",  # Dark Coffee
        "REN": "#9c6644",  # Khaki Ochre
    },
    # ── Immune: Myeloid vs Lymphoid (Warm Tangerine/Rust vs Soft Chartreuse) ──
    "Immune": {
        "Myeloid": "#e76f51",
        "MON": "#f77f00",
        "DC": "#d97706",
        "MAC": "#c85a17",
        "resMAC": "#9a3412",
        "moMAC": "#ea580c",
        "moMAC-INF": "#fb923c",
        "N": "#b45309",
        "MAST": "#f59e0b",
        "Lymphoid": "#d4e09b",
        "T": "#b5e48c",
        "B": "#99d98c",
        "NK": "#76c893",
        "PL": "#f3c68f",
    },
    # ── Neural (Dark Browns) ──
    "Neural": {
        "NEU": "#4a2810",
        "SC/NEU": "#6f4423",
    },
    # ── Low Quality / Unassigned ──
    "Other": {"low_quality": "#e0e0e0", "unassigned": "#ced4da"},
}

# --- 4. Resolve Active Highlight Set ---
canonical_label_colors = {}
for grp_name, cell_dict in kidney_color_groups.items():
  for cell_lbl, col in cell_dict.items():
    canonical_label_colors[cell_lbl] = col

# Determine target labels to highlight
if highlight_labels is None or highlight_labels == "all":
  active_highlight_set = set(df_umap["Unit"].dropna().unique()) - {
      "low_quality",
      "unassigned",
  }
else:
  active_highlight_set = set()
  for item in highlight_labels:
    if item in kidney_color_groups:
      # If a group name was passed, expand to all labels in that group
      active_highlight_set.update(kidney_color_groups[item].keys())
    else:
      active_highlight_set.add(item)

# Build plot color map: Highlighted labels get canonical colors, others get bg_color
plot_colors = {}
for label in df_umap["Unit"].dropna().unique():
  if label in active_highlight_set:
    plot_colors[label] = canonical_label_colors.get(label, "#1f77b4")
  else:
    plot_colors[label] = bg_color

df_umap["color"] = df_umap["Unit"].map(plot_colors).fillna(bg_color)

# --- 5. Plotting ---
fig, ax = plt.subplots(figsize=(18, 14), dpi=150)

# Plot Background (Off-white) points first
bg_subset = df_umap[df_umap["color"] == bg_color]
if not bg_subset.empty:
  ax.scatter(
      bg_subset["UMAP_1"],
      bg_subset["UMAP_2"],
      c=bg_color,
      label="Other / Unselected",
      alpha=bg_alpha,
      s=s * 1.5,
      edgecolors="none",
      rasterized=True,
  )

# Plot Highlighted points on top with their canonical colors
highlighted_present = [
    lbl
    for lbl in canonical_label_colors
    if lbl in active_highlight_set and lbl in df_umap["Unit"].values
]

for label in highlighted_present:
  subset = df_umap[df_umap["Unit"] == label]
  ax.scatter(
      subset["UMAP_1"],
      subset["UMAP_2"],
      c=[plot_colors[label]],
      label=label,
      alpha=fg_alpha,
      s=s * 2.2,
      edgecolors="none",
      rasterized=True,
  )

# --- 6. Formatting & Dynamic Legend ---
title_str = (
    f"UMAP: {highlight_title}"
    if highlight_title
    else "UMAP: Canonical Kidney Subsets"
)
ax.set_title(title_str, fontsize=22, fontweight="bold", pad=15)
ax.set_xlabel("UMAP 1", fontsize=16, labelpad=10)
ax.set_ylabel("UMAP 2", fontsize=16, labelpad=10)
ax.tick_params(axis="both", which="major", labelsize=12)
ax.grid(False)

ncols = 2 if len(highlighted_present) <= 12 else 3
ax.legend(
    title=(
        highlight_title if highlight_title else "Highlighted Lineages"
    ),
    title_fontsize=14,
    fontsize=11,
    markerscale=3.0,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    ncol=ncols,
    frameon=True,
    framealpha=0.9,
)

plt.tight_layout()

# --- 7. Save Options (Uncomment if needed) ---
if 'output_dir' in locals():
    os.makedirs(output_dir, exist_ok=True)
    suffix = f"_{highlight_title.replace(' ', '_')}" if highlight_title else ""
    plt.savefig(os.path.join(output_dir, f"UMAP{suffix}.png"), dpi=300, bbox_inches='tight', transparent=False)
    plt.savefig(os.path.join(output_dir, f"UMAP{suffix}_trans.png"), dpi=300, bbox_inches='tight', transparent=True)

plt.show()


In [194]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

for col in sam_cols:
  # --- 1. Prepare Data ---
  feature_col = col  # Column to color by (e.g. 'Redox', 'SAM_Glycerols', etc.)
  df_umap = pd.merge(df_2_umap, data_df, on='cell_id')
  # df_umap = pd.merge(df_umap, srs_df[d_cols], on='cell_id')

  if "s" not in locals():
    s = 1.0

  # --- 2. Colormap & Value Scaling Configuration ---
  cmap_name = "magma"  # Continuous colormap ('turbo', 'viridis', 'magma', etc.)
  alpha_val = 0.85

  # Optional: Robust percentile clipping to avoid outliers washing out contrast
  # Set to (None, None) for raw min/max, or e.g. (1, 99) for 1st/99th percentiles
  vmin_pct, vmax_pct = 0,99  # e.g., 1, 99

  valid_vals = df_umap[feature_col].dropna()
  vmin = (
      np.nanpercentile(valid_vals, vmin_pct)
      if vmin_pct is not None
      else valid_vals.min()
  )
  vmax = (
      np.nanpercentile(valid_vals, vmax_pct)
      if vmax_pct is not None
      else valid_vals.max()
  )

  # Optional: Sort points by value so higher-intensity cells are visible on top
  sort_by_value = (
      True  # Standard in single-cell/spatial plots (like scanpy/Seurat)
  )
  if sort_by_value:
    df_umap = df_umap.sort_values(
        by=feature_col, na_position="first", ascending=True
    )

  # --- 3. Plotting ---
  fig, ax = plt.subplots(figsize=(18, 14), dpi=150)

  # A. Plot any NaN / missing cells in off-white background first
  nan_subset = df_umap[df_umap[feature_col].isna()]
  if not nan_subset.empty:
    ax.scatter(
        nan_subset["UMAP_1"],
        nan_subset["UMAP_2"],
        c="#e5e5e5",
        label="NaN / Unmeasured",
        alpha=0.35,
        s=s * 1.5,
        edgecolors="none",
        rasterized=True,
    )

  # B. Plot continuous values with the 'turbo' colormap
  val_subset = df_umap[df_umap[feature_col].notna()]
  scatter = ax.scatter(
      val_subset["UMAP_1"],
      val_subset["UMAP_2"],
      c=val_subset[feature_col],
      cmap=cmap_name,
      vmin=vmin,
      vmax=vmax,
      alpha=alpha_val,
      s=s * 2.2,
      edgecolors="none",
      rasterized=True,
  )  
  
  if 'SAM_' in feature_col:
    feature_col = feature_col.replace('SAM_',"")



  # --- 4. Colorbar Integration ---
  cbar = fig.colorbar(scatter, ax=ax, fraction=0.035, pad=0.02)
  cbar.set_label(feature_col, fontsize=16, fontweight="bold", labelpad=12)
  cbar.ax.tick_params(labelsize=12)
  cbar.outline.set_visible(False)

  # --- 5. Formatting ---
  ax.set_title(f"UMAP: {feature_col}", fontsize=22, fontweight="bold", pad=15)
  ax.set_xlabel("UMAP 1", fontsize=16, labelpad=10)
  ax.set_ylabel("UMAP 2", fontsize=16, labelpad=10)
  ax.tick_params(axis="both", which="major", labelsize=12)
  ax.grid(False)

  # Optional legend for NaNs if present
  if not nan_subset.empty:
    ax.legend(
        loc="upper left",
        fontsize=11,
        markerscale=3.0,
        frameon=True,
        framealpha=0.9,
    )

  plt.tight_layout()




  # --- 6. Save Options ---
  if "output_dir" in locals():
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(
        os.path.join(output_dir, f"UMAP_{feature_col}.png"),
        dpi=300,
        bbox_inches="tight",
        transparent=False,
    )
    plt.savefig(
        os.path.join(output_dir, f"UMAP_{feature_col}_trans.png"),
        dpi=300,
        bbox_inches="tight",
        transparent=True,
    )

  # plt.show()
  plt.close()


In [189]:
data_df = pd.merge(data_df, srs_df[d_cols], how='left', on=['cell_id', 'Unit', 'Source_ID'])

In [195]:
data_df.columns

Index(['Unit', 'Source_ID', 'cell_id', 'CH1', 'CH2', 'CH3', 'CH4', 'CH5',
       'CH6', 'CH7', 'CH8', 'CH9', 'CH10', 'CH11', 'CH12', 'CH13', 'CH14',
       'CH15', 'CH16', 'CH17', 'CH18', 'CH19', 'CH20', 'CH21', 'CH22', 'CH23',
       'CH24', 'CH25', 'CH26', 'CH27', 'CH28', 'CH29', 'CH30', 'CH31', 'CH32',
       'CH33', 'CH34', 'CH35', 'CH36', 'CH37', 'CH38', 'CH39', 'CH40', 'CH41',
       'CH42', 'CH43', 'CH44', 'CH45', 'CH46', 'CH47', 'CH48', 'CH49', 'CH50',
       'CH51', 'CH52', 'CH53', 'CH54', 'CH55', 'CH56', 'CH57', 'CH58', 'CH59',
       'CH60', 'CH61', 'Redox', 'Sample_Id', 'SAM_Sterols', 'SAM_Cardiolipins',
       'SAM_Phosphatidylcholines', 'SAM_Phosphatidylethanolamines',
       'SAM_Lysophosphatidate', 'SAM_Glycerols', 'SAM_SFA', 'SAM_PUFA',
       'SAM_Carnitines', 'SAM_Ceramides', 'SAM_Phosphatidylserines',
       'SAM_Phosphatidylinositols', 'SAM_Phosphatidylglycerols',
       'SAM_Glucosylceramides'],
      dtype='object')

In [196]:
data_df.to_csv(os.path.join(output_dir, "hubmap_round1_cell_mean_metabolism_features.csv"), index=False)

In [ ]:
def macro_idx(wavenumber, num_samp, wavenum_1=2700, wavenum_2=3100):
    idx = int(((wavenumber - wavenum_1) / (wavenum_2 - wavenum_1)) * num_samp)
    return idx
shift = 20
num_samp = 61
wn_1 = 2700
wn_2 = 3100
unsat_idx = macro_idx(3050 + shift, num_samp, wn_1, wn_2)
protein_idx = macro_idx(2930 + shift, num_samp, wn_1, wn_2)
sat_idx = macro_idx(2880 + shift, num_samp, wn_1, wn_2)
lipid_idx = macro_idx(2850 + shift, num_samp, wn_1, wn_2)

print(unsat_idx, protein_idx, sat_idx, lipid_idx)

In [ ]:
protein_ch = data_df[ch_cols[protein_idx-2:protein_idx+2]].max(axis=1)
lipid_ch = data_df[ch_cols[lipid_idx-2:lipid_idx+2]].max(axis=1)
sat_ch = data_df[ch_cols[sat_idx-2:sat_idx+2]].max(axis=1)
unsat_ch = data_df[ch_cols[unsat_idx-2:unsat_idx+2]].max(axis=1)

lip_unsat = np.maximum(unsat_ch / (sat_ch + unsat_ch), 0)
lip_pro = np.maximum(lipid_ch / (lipid_ch + protein_ch),0)

data_df['Lipid Unsaturation'] = lip_unsat
data_df['Lipid-Protein Ratio'] = lip_pro

In [ ]:
np.maximum(protein_ch / lipid_ch,0)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df_umap = df_2_umap

# --- 2. Define labels to highlight and colors ---
highlight_labels = ['POD', 'PEC', 'PT', 'DTL', 'ATL', 'TAL', 'DCT', 'CNT', 'PC', 'IC'] # Replace with your specific labels
highlight_title = "Epithelial"
# color_palette = ['#e63946', '#2a9d8f']     # Specify colors for your highlighted labels
color_palette = sns.color_palette('husl', len(highlight_labels))

# Create a dictionary mapping specific labels to colors, and all others to gray
label_colors = {label: color for label, color in zip(highlight_labels, color_palette)}
df_umap['color'] = df_umap['Unit'].map(label_colors).fillna('#d3d3d3') # '#d3d3d3' is light gray

# --- 3. Plotting ---
plt.figure(figsize=(16, 12))

# Plot the gray (unhighlighted) points first so they are in the background
sns.scatterplot(
    data=df_umap[df_umap['color'] == '#d3d3d3'], 
    x='UMAP_1', y='UMAP_2', 
    color='#d3d3d3', 
    alpha=0.5, # Adjust transparency so dense areas don't clump
    label='Other',
    s=s*2
)

# Plot the highlighted points on top
for label, color in label_colors.items():
    subset = df_umap[df_umap['Unit'] == label]
    sns.scatterplot(
        data=subset, 
        x='UMAP_1', y='UMAP_2', 
        color=color, 
        label=label,
        alpha=1.0,
        s=s*2
    )

plt.legend(title=highlight_title, markerscale=2)
plt.title(f'UMAP: {highlight_title}', fontsize=20, fontweight='bold')
plt.xlabel('UMAP 1', fontsize=16)
plt.ylabel('UMAP 2', fontsize=16)
# plt.savefig(os.path.join(output_dir,f"UMAP_{highlight_title}.png"),dpi=300, transparent=False)
# plt.savefig(os.path.join(output_dir,f"UMAP_{highlight_title}_trans.png"),dpi=300, transparent=True)
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- 2. Define labels to highlight and colors ---
highlight_labels = ['POD', 'PEC', 'PT', 'DTL', 'ATL', 'TAL', 'DCT', 'CNT', 'PC', 'IC'] # Replace with your specific labels
highlight_title = "Epithelial"
# color_palette = ['#e63946', '#2a9d8f']     # Specify colors for your highlighted labels
color_palette = sns.color_palette('husl', len(highlight_labels))

# Create a dictionary mapping specific labels to colors, and all others to gray
label_colors = {label: color for label, color in zip(highlight_labels, color_palette)}
df_normalized_umap['color'] = df_normalized_umap['Unit'].map(label_colors).fillna('#d3d3d3') # '#d3d3d3' is light gray

# --- 3. Plotting ---
plt.figure(figsize=(16, 12))

# Plot the gray (unhighlighted) points first so they are in the background
sns.scatterplot(
    data=df_normalized_umap[df_normalized_umap['color'] == '#d3d3d3'], 
    x='UMAP_1', y='UMAP_2', 
    color='#d3d3d3', 
    alpha=0.5, # Adjust transparency so dense areas don't clump
    label='Other',
    s=s*2
)

# Plot the highlighted points on top
for label, color in label_colors.items():
    subset = df_normalized_umap[df_normalized_umap['Unit'] == label]
    sns.scatterplot(
        data=subset, 
        x='UMAP_1', y='UMAP_2', 
        color=color, 
        label=label,
        alpha=1.0,
        s=s*2
    )

plt.legend(title=highlight_title, markerscale=2)
plt.title(f'UMAP: {highlight_title}', fontsize=20, fontweight='bold')
plt.xlabel('UMAP 1', fontsize=16)
plt.ylabel('UMAP 2', fontsize=16)
# plt.savefig(os.path.join(output_dir,f"UMAP_{highlight_title}.png"),dpi=300, transparent=False)
# plt.savefig(os.path.join(output_dir,f"UMAP_{highlight_title}_trans.png"),dpi=300, transparent=True)
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- 2. Define labels to highlight and colors ---
highlight_labels = ['altPOD', 'altPEC', 'altPT', 'altDTL', 'altATL', 'altTAL', 'altDCT', 'altCNT', 'altPC', 'altIC'] # Replace with your specific labels
highlight_title = "altEpithelial"
# color_palette = ['#e63946', '#2a9d8f']     # Specify colors for your highlighted labels
color_palette = sns.color_palette('husl', len(highlight_labels))

# Create a dictionary mapping specific labels to colors, and all others to gray
label_colors = {label: color for label, color in zip(highlight_labels, color_palette)}
df_umap['color'] = df_umap['Unit'].map(label_colors).fillna('#d3d3d3') # '#d3d3d3' is light gray

# --- 3. Plotting ---
plt.figure(figsize=(16, 12))

# Plot the gray (unhighlighted) points first so they are in the background
sns.scatterplot(
    data=df_umap[df_umap['color'] == '#d3d3d3'], 
    x='UMAP_1', y='UMAP_2', 
    color='#d3d3d3', 
    alpha=0.5, # Adjust transparency so dense areas don't clump
    label='Other',
    s=s*2
)

# Plot the highlighted points on top
for label, color in label_colors.items():
    subset = df_umap[df_umap['Unit'] == label]
    sns.scatterplot(
        data=subset, 
        x='UMAP_1', y='UMAP_2', 
        color=color, 
        label=label,
        alpha=1.0,
        s=s*2
    )

plt.legend(title=highlight_title, markerscale=2)
plt.title(f'UMAP: {highlight_title}', fontsize=20, fontweight='bold')
plt.xlabel('UMAP 1', fontsize=16)
plt.ylabel('UMAP 2', fontsize=16)
# plt.savefig(os.path.join(output_dir,f"UMAP_{highlight_title}.png"),dpi=300, transparent=False)
# plt.savefig(os.path.join(output_dir,f"UMAP_{highlight_title}_trans.png"),dpi=300, transparent=True)
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- 2. Define labels to highlight and colors ---
highlight_labels = ['FIB', 'aFIB', 'pvFIB', 'infFIB', 'MYOF', 'MC', 'VSM/P'] # Replace with your specific labels
highlight_title = "Stromal"
# color_palette = ['#e63946', '#2a9d8f']     # Specify colors for your highlighted labels
color_palette = sns.color_palette('husl', len(highlight_labels))

# Create a dictionary mapping specific labels to colors, and all others to gray
label_colors = {label: color for label, color in zip(highlight_labels, color_palette)}
df_umap['color'] = df_umap['Unit'].map(label_colors).fillna('#d3d3d3') # '#d3d3d3' is light gray

# --- 3. Plotting ---
plt.figure(figsize=(16, 12))

# Plot the gray (unhighlighted) points first so they are in the background
sns.scatterplot(
    data=df_umap[df_umap['color'] == '#d3d3d3'], 
    x='UMAP_1', y='UMAP_2', 
    color='#d3d3d3', 
    alpha=0.5, # Adjust transparency so dense areas don't clump
    label='Other',
    s=s*2
)

# Plot the highlighted points on top
for label, color in label_colors.items():
    subset = df_umap[df_umap['Unit'] == label]
    sns.scatterplot(
        data=subset, 
        x='UMAP_1', y='UMAP_2', 
        color=color, 
        label=label,
        alpha=1.0,
        s=s*2
    )

plt.legend(title=highlight_title, markerscale=2)
plt.title(f'UMAP: {highlight_title}', fontsize=20, fontweight='bold')
plt.xlabel('UMAP 1', fontsize=16)
plt.ylabel('UMAP 2', fontsize=16)
plt.savefig(os.path.join(output_dir,f"UMAP_{highlight_title}.png"),dpi=300, transparent=False)
plt.savefig(os.path.join(output_dir,f"UMAP_{highlight_title}_trans.png"),dpi=300, transparent=True)
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- 2. Define labels to highlight and colors ---
highlight_labels = ['EC-GC', 'EC'] # Replace with your specific labels
highlight_title = "Endothelial"
# color_palette = ['#e63946', '#2a9d8f']     # Specify colors for your highlighted labels
color_palette = sns.color_palette('husl', len(highlight_labels))

# Create a dictionary mapping specific labels to colors, and all others to gray
label_colors = {label: color for label, color in zip(highlight_labels, color_palette)}
df_umap['color'] = df_umap['Unit'].map(label_colors).fillna('#d3d3d3') # '#d3d3d3' is light gray

# --- 3. Plotting ---
plt.figure(figsize=(16, 12))

# Plot the gray (unhighlighted) points first so they are in the background
sns.scatterplot(
    data=df_umap[df_umap['color'] == '#d3d3d3'], 
    x='UMAP_1', y='UMAP_2', 
    color='#d3d3d3', 
    alpha=0.5, # Adjust transparency so dense areas don't clump
    label='Other',
    s=s*2
)

# Plot the highlighted points on top
for label, color in label_colors.items():
    subset = df_umap[df_umap['Unit'] == label]
    sns.scatterplot(
        data=subset, 
        x='UMAP_1', y='UMAP_2', 
        color=color, 
        label=label,
        alpha=1.0,
        s=s*2
    )

plt.legend(title=highlight_title, markerscale=2)
plt.title(f'UMAP: {highlight_title}', fontsize=20, fontweight='bold')
plt.xlabel('UMAP 1', fontsize=16)
plt.ylabel('UMAP 2', fontsize=16)
plt.savefig(os.path.join(output_dir,f"UMAP_{highlight_title}.png"),dpi=300, transparent=False)
plt.savefig(os.path.join(output_dir,f"UMAP_{highlight_title}_trans.png"),dpi=300, transparent=True)
plt.show()


In [ ]:
from statsmodels.stats import weightstats as sms


value = 'Redox'
group = 'Source_ID'
plot_data_df = data_df.dropna(subset=[value])
# groups = [data_df[data_df['Source_ID'] == source][value] for source in data_df['Source_ID'].unique()]
result = sm.stats.anova_oneway(plot_data_df[value], groups=plot_data_df[group], use_var='equal')
exact_p_value = 1 - stats.f.cdf(result.statistic, result.df_num, result.df_denom)

tukey = pairwise_tukeyhsd(endog=plot_data_df[value], groups=plot_data_df[group], alpha=0.05)
tukey_summary = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
pvalues = tukey.pvalues

group1_col = tukey_summary.iloc[:, 0]
group2_col = tukey_summary.iloc[:, 1]


pairs= [(g1, g2) for g1, g2 in zip(group1_col, group2_col)]
for pair in pairs:
    group1 = plot_data_df[plot_data_df[group]==pair[0]][value]
    group2 = plot_data_df[plot_data_df[group]==pair[1]][value]

    v=cohens_d_threshold = 0.5

    # 3. Calculate Pooled Standard Deviation manually
    n_A, n_B = len(group1), len(group2)
    var_A, var_B = np.var(group1, ddof=1), np.var(group2, ddof=1)

    pooled_sd = np.sqrt(((n_A - 1) * var_A + (n_B - 1) * var_B) / (n_A + n_B - 2))

    # 4. Convert Cohen's d to Raw Data Units
    # This scales the 0.2 effect size specifically to your data's variance
    raw_bound = cohens_d_threshold * pooled_sd
    print(f"{pair} - raw bounds: {raw_bound}")
    low_bound = -raw_bound
    high_bound = raw_bound
    # Run the Two One-Sided Tests (TOST)
    p_val, sm_tstat, _ = sms.ttost_ind(group1, group2, low_bound, high_bound)

    print(f"TOST Equivalence p-value: {p_val:.4f}")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.stats.weightstats as sms
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statannotations.Annotator import Annotator
def plot_violin_stats(
    data_df, 
    value, 
    compare_col, 
    compare_levels, 
    facet_col=None,
    facet_levels=None, 
    summary_type='mean', 
    output_dir='output',
    title_suffix="",
    ylim=None,
    show_plot=True
):
    """
    Plots group-wise violins with ANOVA, post-hoc Tukey HSD, and TOST equivalence testing.
    Supports two modes:
      1. Faceted mode (when facet_col is provided): Generates a grid of subplots.
      2. Single plot mode (when facet_col is None): Generates a single-axes plot.
    """
    # 1. Establish the color palette on the FULL, original list of compare_levels to ensure cross-image consistency
    palette = sns.color_palette("turbo", len(compare_levels))
    group_palette = dict(zip(compare_levels, palette))
    # Filter levels to those present in the provided dataframe
    selected_levels = [level for level in compare_levels if level in data_df[compare_col].dropna().unique()]
    if not selected_levels:
        raise ValueError(f"None of the requested levels {compare_levels} found in column '{compare_col}'")
    # Determine Facet Values & ordering
    if facet_col is not None:
        if facet_levels is not None:
            # Filter and order based on user-supplied facet_levels
            facet_values = [facet for facet in facet_levels if facet in data_df[facet_col].dropna().unique()]
        else:
            # Default to all unique values found in data
            facet_values = [facet for facet in data_df[facet_col].dropna().unique()]
            
        n_facets = len(facet_values)
        if n_facets == 0:
            raise ValueError(f"No valid facet values from {facet_levels or 'dataframe'} found in column '{facet_col}'")
        
        
        # Grid setup
        n_cols = min(7, n_facets)
        n_rows = int(np.ceil(n_facets / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(2 * n_cols, 6 * n_rows), sharey=False)
        axes = np.atleast_1d(axes).ravel()
        vp_output_dir = os.path.join(output_dir, value.replace(':','-'), f'faceted_by_{facet_col}')
    else:
        # Single plot setup
        facet_values = [None]
        n_facets = 1
        n_cols = 1
        n_rows = 1
        fig, ax = plt.subplots(figsize=(4, 6))
        axes = [ax]
        vp_output_dir = os.path.join(output_dir, value.replace(':','-'))

    sig_counts = []
    # Iterate over facets (or just run once for a single plot)
    for idx, (ax, facet_value) in enumerate(zip(axes, facet_values)):
        try:
            # Filter data for current panel
            if facet_col is not None:
                subset_df = data_df[data_df[facet_col] == facet_value]
                panel_title = f"{facet_value}"
            else:
                subset_df = data_df
                panel_title = f"{title_suffix}" if title_suffix else ""
            plot_data_df = subset_df[subset_df[compare_col].isin(selected_levels)].dropna(subset=[value]).copy()
            if plot_data_df.empty:
                ax.set_axis_off()
                continue
            # Ensure valid levels have at least 2 observations
            df_counts = plot_data_df[compare_col].value_counts()
            valid_levels = [level for level in selected_levels if df_counts.get(level, 0) >= 2]
            plot_data_df = plot_data_df[plot_data_df[compare_col].isin(valid_levels)]
            if plot_data_df[compare_col].nunique() < 2:

                ax.set_axis_off()
                # ax.set_title(f"{panel_title}\n(not enough samples)", fontsize=14)
                continue
            # Ensure strict and consistent ordering
            unit_order = [level for level in selected_levels if level in plot_data_df[compare_col].unique()]
            # ANOVA
            result = sm.stats.anova_oneway(plot_data_df[value], groups=plot_data_df[compare_col], use_var='equal')
            exact_p_value = 1 - stats.f.cdf(result.statistic, result.df_num, result.df_denom)
            # Tukey HSD Post-hoc
            tukey = pairwise_tukeyhsd(endog=plot_data_df[value], groups=plot_data_df[compare_col], alpha=0.05)
            tukey_summary = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
            pvalues = tukey.pvalues.copy()
            pairs = list(zip(tukey_summary.iloc[:, 0], tukey_summary.iloc[:, 1]))
            # TOST Equivalence check
            for i, pair in enumerate(pairs):
                group1 = plot_data_df[plot_data_df[compare_col] == pair[0]][value]
                group2 = plot_data_df[plot_data_df[compare_col] == pair[1]][value]
                cohens_d_threshold = 0.2
                n_A, n_B = len(group1), len(group2)
                var_A, var_B = np.var(group1, ddof=1), np.var(group2, ddof=1)
                pooled_sd = np.sqrt(((n_A - 1) * var_A + (n_B - 1) * var_B) / (n_A + n_B - 2))
                raw_bound = cohens_d_threshold * pooled_sd
                low_bound = -raw_bound
                high_bound = raw_bound
                p_val, _, _ = sms.ttost_ind(group1, group2, low_bound, high_bound)
                if p_val < 0.05:
                    pvalues[i] = 1.0
            # Violin Plotting (using the persistent group_palette lookup)
            sns.boxplot(
                data=plot_data_df,
                x=compare_col,
                y=value,
                hue=compare_col if facet_col is not None else None,
                order=unit_order,
                palette=group_palette,
                dodge=False,
                legend=False,
                ax=ax,
                showfliers=False,
                linewidth=3
            )
            # if facet_col is None:
            #     # Annotate statistical summaries (mean or median)
            #     if summary_type == 'mean':
            #         summary_vals = plot_data_df.groupby(compare_col)[value].mean().reindex(unit_order)
            #         fmt = ".3f"
            #     else:
            #         summary_vals = plot_data_df.groupby(compare_col)[value].median().reindex(unit_order)
            #         fmt = ".3f"
            #     for i, val in enumerate(summary_vals):
            #         if not np.isnan(val):
            #             ax.text(
            #                 i, val, f'{val:{fmt}}',fontsize=14,
            #                 horizontalalignment='center',
            #                 color='black', weight='bold',
            #                 bbox=dict(facecolor='white', alpha=0.7)
                    # )
            # Mask non-significant pairs
            alpha = 0.05
            pvalues = np.asarray(pvalues)
            mask = pvalues < alpha
            pairs_sig = [p for p, m in zip(pairs, mask) if m]
            pvalues_sig = pvalues[mask]
            sig_counts.append(len(pairs_sig))
            # Annotations with dynamic placement
            if pairs_sig:
                annotator = Annotator(ax, pairs_sig, data=plot_data_df, x=compare_col, y=value, order=unit_order)
                annotator.configure(text_format="star", loc="outside")
                annotator.set_pvalues(pvalues_sig)
                annotator.annotate()
            sns.despine(ax=ax)
            ax.set_xlabel('', fontsize=14)
            # Y label logic: display only on the leftmost column of the grid in faceted mode, or always in single mode
            if facet_col is not None:
                is_left_edge = (idx % n_cols == 0)
                ax.set_ylabel(value.replace('_', ' ').replace('to', '/').replace('RF','').replace('SAM','') if is_left_edge else '', fontsize=24, fontweight='bold')
                ax.minorticks_on()
                ax.tick_params(which='minor', direction='out', length=5, width=1, color='gray')
                ax.tick_params(which='major', direction='out', length=8, width=1, color='black')
                ax.tick_params(axis='x', which='minor', bottom=False)
                ax.tick_params(axis='both', labelsize=18)
            else:
                ax.set_ylabel(value.replace('_', ' ').replace('to', '/').replace('RF','').replace('SAM',''), fontsize=24, fontweight='bold')
                ax.tick_params(axis='both', labelsize=18)
        except Exception as e:
            ax.set_axis_off()
            ax.set_title(f"{facet_value if facet_value else 'Plot'}\nfailed: {e}", fontsize=12)

    
    # 3. Synchronize Y-Limits Automatically Across Active Plots
    all_ylims = [ax.get_ylim() for ax in axes[:n_facets] if ax.axison]
    if all_ylims:
        global_ymin = min(y[0] for y in all_ylims)
        global_ymax = max(y[1] for y in all_ylims)
        
        final_ylim = ylim if ylim is not None else (global_ymin, global_ymax)
        for ax in axes[:n_facets]:
            if ax.axison:
                ax.set_ylim(final_ylim)
                
                # In faceted mode, hide y-tick labels for non-left-edge plots to mimic sharey behavior
                if facet_col is not None:
                    # Find index of this axis
                    ax_idx = list(axes).index(ax)
                    if ax_idx % n_cols != 0:
                        ax.tick_params(labelleft=False)
            

    # 2. Dynamic Padding & Layout Calculations
    max_sig_bars = max(sig_counts) if sig_counts else 0
    title_y = 1.2 + (0.06 * max_sig_bars)
    if facet_col is not None:
        # Layout metrics for the grid layout
        h_pad_val = 1.5 + (1.2 * max_sig_bars)
        rect_top = max(0.6, 0.98 - (0.03 * max_sig_bars))
        suptitle_y = max(1.5, 1.1 - (0.01 * max_sig_bars))
    else:
        # Layout metrics for a single plot panel to prevent the suptitle from being pushed too high
        h_pad_val = 1.5 + (1.2 * max_sig_bars)
        rect_top = max(0.6, 0.98 - (0.03 * max_sig_bars))
        suptitle_y = max(1, 1.01 - (0.001 * max_sig_bars))

        
    # Apply dynamic titles and layouts to valid active subplots
    for idx, (ax, facet_value) in enumerate(zip(axes[:n_facets], facet_values)):
        if ax.axison: # Only edit layout for panels that didn't fail/turn off
            if facet_col is not None:
                ax.set_title(f'{facet_value}', fontsize=24, y=title_y, color='black')
            else:
                ax.set_title(f"ANOVA p-value: {exact_p_value:.2e}", fontsize=16, color='gray', y=title_y, pad=10)

    # Turn off unused axes in the grid
    for ax in axes[n_facets:]:
        ax.set_axis_off()

    # Dynamic top margin configuration
    fig.suptitle(
        # f"{value.replace('_', ' ').replace('to', '/').replace('RF','').replace('SAM','')}" + (f": {title_suffix}" if (not facet_col and title_suffix) else ""),
        "",
        fontsize=22,
        fontweight='bold',
        y=suptitle_y,
        x=0.56 if facet_col is None else 0.5
    )
    os.makedirs(vp_output_dir, exist_ok=True)
    if facet_col is not None:
        fig.tight_layout(rect=[0, 0, 1, rect_top], h_pad=h_pad_val)
        save_name = f'faceted_vplot_{value.replace(":", "-")}_{compare_levels[0]}_{len(compare_levels)}.png'
    else:
        fig.tight_layout(rect=[0, 0, 1, rect_top])
        suffix_clean = title_suffix.replace("/", "").replace(" ", "_") if title_suffix else ""
        save_name = f'vplot_{value.replace(":", "-")}_{suffix_clean}.png'

    plt.tight_layout()
    fig.savefig(os.path.join(vp_output_dir, save_name), dpi=300, transparent=True, bbox_inches='tight')
    print(f"Violin plot saved in {vp_output_dir} as {save_name}")
    if show_plot:
        plt.show()
    plt.close(fig)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.stats.weightstats as sms
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statannotations.Annotator import Annotator

def plot_violin_stats(
    data_df, 
    value, 
    compare_col, 
    compare_levels, 
    facet_col=None,
    facet_levels=None, 
    summary_type='mean', 
    output_dir='output',
    title_suffix="",
    ylim=None, # Acts as x-limits (xlim) when orient='h'
    show_plot=True,
    orient='v' # Added parameter: 'v' for vertical, 'h' for horizontal
):
    """
    Plots group-wise violins/boxplots with ANOVA, post-hoc Tukey HSD, and TOST equivalence testing.
    Supports both vertical and horizontal layouts.
    """
    # 1. Establish the color palette on the FULL original compare_levels
    palette = sns.color_palette("turbo", len(compare_levels))
    group_palette = dict(zip(compare_levels, palette))
    
    # Filter levels to those present in the provided dataframe
    selected_levels = [level for level in compare_levels if level in data_df[compare_col].dropna().unique()]
    if not selected_levels:
        raise ValueError(f"None of the requested levels {compare_levels} found in column '{compare_col}'")
        
    # Determine Facet Values & ordering
    if facet_col is not None:
        if facet_levels is not None:
            facet_values = [facet for facet in facet_levels if facet in data_df[facet_col].dropna().unique()]
        else:
            facet_values = [facet for facet in data_df[facet_col].dropna().unique()]
            
        n_facets = len(facet_values)
        if n_facets == 0:
            raise ValueError(f"No valid facet values from {facet_levels or 'dataframe'} found in column '{facet_col}'")
        
        # Grid setup
        n_cols = min(7, n_facets)
        n_rows = int(np.ceil(n_facets / n_cols))
        
        # Adjust grid figure size ratios based on orientation
        if orient == 'v':
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(2 * n_cols, 6 * n_rows), sharey=False)
        else:
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 2 * n_rows), sharex=False)
            
        axes = np.atleast_1d(axes).ravel()
        vp_output_dir = os.path.join(output_dir, value.replace(':','-'), f'faceted_by_{facet_col}')
    else:
        # Single plot setup
        facet_values = [None]
        n_facets = 1
        n_cols = 1
        n_rows = 1
        # Set size based on orientation
        figsize = (4, 6) if orient == 'v' else (6, 4)
        fig, ax = plt.subplots(figsize=figsize)
        axes = [ax]
        vp_output_dir = os.path.join(output_dir, value.replace(':','-'))

    sig_counts = []
    
    for idx, (ax, facet_value) in enumerate(zip(axes, facet_values)):
        try:
            # Filter data for current panel
            if facet_col is not None:
                subset_df = data_df[data_df[facet_col] == facet_value]
                panel_title = f"{facet_value}"
            else:
                subset_df = data_df
                panel_title = f"{title_suffix}" if title_suffix else ""
                
            plot_data_df = subset_df[subset_df[compare_col].isin(selected_levels)].dropna(subset=[value]).copy()
            if plot_data_df.empty:
                ax.set_axis_off()
                continue
                
            # Ensure valid levels have at least 2 observations
            df_counts = plot_data_df[compare_col].value_counts()
            valid_levels = [level for level in selected_levels if df_counts.get(level, 0) >= 2]
            plot_data_df = plot_data_df[plot_data_df[compare_col].isin(valid_levels)]
            
            if plot_data_df[compare_col].nunique() < 2:
                ax.set_axis_off()
                continue
                
            # Ensure strict and consistent ordering
            unit_order = [level for level in selected_levels if level in plot_data_df[compare_col].unique()]
            
            # ANOVA
            result = sm.stats.anova_oneway(plot_data_df[value], groups=plot_data_df[compare_col], use_var='equal')
            exact_p_value = 1 - stats.f.cdf(result.statistic, result.df_num, result.df_denom)
            
            # Tukey HSD Post-hoc
            tukey = pairwise_tukeyhsd(endog=plot_data_df[value], groups=plot_data_df[compare_col], alpha=0.05)
            tukey_summary = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
            pvalues = tukey.pvalues.copy()
            pairs = list(zip(tukey_summary.iloc[:, 0], tukey_summary.iloc[:, 1]))
            
            # TOST Equivalence check
            for i, pair in enumerate(pairs):
                group1 = plot_data_df[plot_data_df[compare_col] == pair[0]][value]
                group2 = plot_data_df[plot_data_df[compare_col] == pair[1]][value]
                cohens_d_threshold = 0.2
                n_A, n_B = len(group1), len(group2)
                var_A, var_B = np.var(group1, ddof=1), np.var(group2, ddof=1)
                pooled_sd = np.sqrt(((n_A - 1) * var_A + (n_B - 1) * var_B) / (n_A + n_B - 2))
                raw_bound = cohens_d_threshold * pooled_sd
                low_bound = -raw_bound
                high_bound = raw_bound
                p_val, _, _ = sms.ttost_ind(group1, group2, low_bound, high_bound)
                if p_val < 0.05:
                    pvalues[i] = 1.0
            
            # Box Plotting (swapping X and Y mapping depending on orientation)
            if orient == 'v':
                sns.boxplot(
                    data=plot_data_df,
                    x=compare_col,
                    y=value,
                    hue=compare_col if facet_col is not None else None,
                    order=unit_order,
                    palette=group_palette,
                    dodge=False,
                    legend=False,
                    ax=ax,
                    showfliers=False,
                    linewidth=3
                )
            else:
                sns.boxplot(
                    data=plot_data_df,
                    x=value,
                    y=compare_col,
                    hue=compare_col if facet_col is not None else None,
                    order=unit_order,
                    palette=group_palette,
                    dodge=False,
                    legend=False,
                    ax=ax,
                    showfliers=False,
                    linewidth=3
                )

            # Mask non-significant pairs
            alpha = 0.05
            pvalues = np.asarray(pvalues)
            mask = pvalues < alpha
            pairs_sig = [p for p, m in zip(pairs, mask) if m]
            pvalues_sig = pvalues[mask]
            sig_counts.append(len(pairs_sig))
            
            # Annotations with dynamic placement
            if pairs_sig:
                if orient == 'v':
                    annotator = Annotator(ax, pairs_sig, data=plot_data_df, x=compare_col, y=value, order=unit_order, orient='v')
                else:
                    annotator = Annotator(ax, pairs_sig, data=plot_data_df, x=value, y=compare_col, order=unit_order, orient='h')
                annotator.configure(text_format="star", loc="outside")
                annotator.set_pvalues(pvalues_sig)
                annotator.annotate()
                
            sns.despine(ax=ax)
            
            # Axis formatting based on orientation
            label_text = value.replace('_', ' ').replace('to', '/').replace('RF','').replace('SAM','')
            
            if orient == 'v':
                ax.set_xlabel('', fontsize=14)
                if facet_col is not None:
                    is_left_edge = (idx % n_cols == 0)
                    ax.set_ylabel(label_text if is_left_edge else '', fontsize=24, fontweight='bold')
                    ax.minorticks_on()
                    ax.tick_params(which='minor', direction='out', length=5, width=1, color='gray')
                    ax.tick_params(which='major', direction='out', length=8, width=1, color='black')
                    ax.tick_params(axis='x', which='minor', bottom=False)
                    ax.tick_params(axis='both', labelsize=18)
                else:
                    ax.set_ylabel(label_text, fontsize=24, fontweight='bold')
                    ax.tick_params(axis='both', labelsize=18)
            else:
                ax.set_ylabel('', fontsize=14)
                if facet_col is not None:
                    # For horizontal facet layout, only show X label on bottom rows
                    is_bottom_row = (idx >= n_facets - n_cols)
                    ax.set_xlabel(label_text if is_bottom_row else '', fontsize=24, fontweight='bold')
                    ax.minorticks_on()
                    ax.tick_params(which='minor', direction='out', length=5, width=1, color='gray')
                    ax.tick_params(which='major', direction='out', length=8, width=1, color='black')
                    ax.tick_params(axis='y', which='minor', left=False) # Turn off minor ticks for discrete categories
                    ax.tick_params(axis='both', labelsize=18)
                else:
                    ax.set_xlabel(label_text, fontsize=24, fontweight='bold')
                    ax.tick_params(axis='both', labelsize=18)
                    
        except Exception as e:
            ax.set_axis_off()
            ax.set_title(f"{facet_value if facet_value else 'Plot'}\nfailed: {e}", fontsize=12)

    # 3. Synchronize Limits Across Active Plots
    if orient == 'v':
        all_ylims = [ax.get_ylim() for ax in axes[:n_facets] if ax.axison]
        if all_ylims:
            global_ymin = min(y[0] for y in all_ylims)
            global_ymax = max(y[1] for y in all_ylims)
            final_ylim = ylim if ylim is not None else (global_ymin, global_ymax)
            for ax in axes[:n_facets]:
                if ax.axison:
                    ax.set_ylim(final_ylim)
                    if facet_col is not None:
                        ax_idx = list(axes).index(ax)
                        if ax_idx % n_cols != 0:
                            ax.tick_params(labelleft=False)
    else:
        all_xlims = [ax.get_xlim() for ax in axes[:n_facets] if ax.axison]
        if all_xlims:
            global_xmin = min(x[0] for x in all_xlims)
            global_xmax = max(x[1] for x in all_xlims)
            final_xlim = ylim if ylim is not None else (global_xmin, global_xmax)
            for ax in axes[:n_facets]:
                if ax.axison:
                    ax.set_xlim(final_xlim)
                    if facet_col is not None:
                        ax_idx = list(axes).index(ax)
                        # Hide category labels on Y axis for plots not on the left edge
                        if ax_idx % n_cols != 0:
                            ax.tick_params(labelleft=False)

    # 2. Dynamic Padding & Layout Calculations
    max_sig_bars = max(sig_counts) if sig_counts else 0
    title_y = 1.2 + (0.06 * max_sig_bars)
    if facet_col is not None:
        h_pad_val = 1.5 + (1.2 * max_sig_bars)
        rect_top = max(0.6, 0.98 - (0.03 * max_sig_bars))
        suptitle_y = max(1.5, 1.1 - (0.01 * max_sig_bars))
    else:
        h_pad_val = 1.5 + (1.2 * max_sig_bars)
        rect_top = max(0.6, 0.98 - (0.03 * max_sig_bars))
        suptitle_y = max(1, 1.01 - (0.001 * max_sig_bars))

    # Apply dynamic titles and layouts to valid active subplots
    for idx, (ax, facet_value) in enumerate(zip(axes[:n_facets], facet_values)):
        if ax.axison:
            if facet_col is not None:
                ax.set_title(f'{facet_value}', fontsize=24, y=title_y, color='black')
            else:
                ax.set_title(f"ANOVA p-value: {exact_p_value:.2e}", fontsize=16, color='gray', y=title_y, pad=10)

    for ax in axes[n_facets:]:
        ax.set_axis_off()

    fig.suptitle(
        "",
        fontsize=22,
        fontweight='bold',
        y=suptitle_y,
        x=0.56 if facet_col is None else 0.5
    )
    
    os.makedirs(vp_output_dir, exist_ok=True)
    if facet_col is not None:
        fig.tight_layout(rect=[0, 0, 1, rect_top], h_pad=h_pad_val)
        save_name = f'faceted_hplot_{value.replace(":", "-")}_{compare_levels[0]}_{len(compare_levels)}.png' if orient == 'h' else f'faceted_vplot_{value.replace(":", "-")}_{compare_levels[0]}_{len(compare_levels)}.png'
    else:
        fig.tight_layout(rect=[0, 0, 1, rect_top])
        suffix_clean = title_suffix.replace("/", "").replace(" ", "_") if title_suffix else ""
        save_name = f'hplot_{value.replace(":", "-")}_{suffix_clean}.png' if orient == 'h' else f'vplot_{value.replace(":", "-")}_{suffix_clean}.png'

    plt.tight_layout()
    fig.savefig(os.path.join(vp_output_dir, save_name), dpi=300, transparent=True, bbox_inches='tight')
    print(f"Plot saved in {vp_output_dir} as {save_name}")
    if show_plot:
        plt.show()
    plt.close(fig)


In [ ]:
for unit in data_df['Unit'].unique()[:]:
    # Subset dataframe to run the logic on each single unit individually
    unit_df = data_df[data_df['Unit'] == unit]
    
    plot_violin_stats(
        data_df=unit_df,
        value='Collagen_Thickness',
        compare_col='Source_ID',
        compare_levels=['REF', 'AKI', 'CKD'],
        facet_col=None,       # Single plot mode
        summary_type='median', # Displays medians with .2f precision
        output_dir=output_dir,
        title_suffix=unit,
        show_plot=False
    )

In [ ]:
subset_df = data_df[data_df['Unit'].isin(unit_order)]

plot_violin_stats(
    data_df=subset_df,
    value='Collagen_Coherence',
    compare_col='Source_ID',
    compare_levels=['REF', 'AKI', 'CKD'],
    facet_col=None,       # Single plot mode
    summary_type='median', # Displays medians with .2f precision
    output_dir=output_dir,
    title_suffix='all',
    show_plot=True,
    orient='h'
)

In [ ]:
['Myeloid', 'MON', 'DC','resMAC', 'moMAC-INF', 'N',]


In [ ]:
value = 'Redox'
plot_violin_stats(
    data_df=data_df,
    value=value,
    compare_col='Source_ID',
    compare_levels=['REF', 'AKI', 'CKD'],
    facet_col='Unit',
    facet_levels=['POD', 'PT', 'altPT', 'DCT', 'altDCT', 'CNT', 'altCNT', 'TAL', 'altTAL'],
    summary_type='median',  # Displays means with .3f precision
    output_dir=output_dir,
    show_plot=True
)



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.patches import Patch

def plot_lipid_composition(
    data_df, 
    sam_cols, 
    group_col='Source_ID', 
    unit_col='Unit', 
    units=None, 
    label_color_dict=None
):
    """
    Plots a row of pie charts (one per group) showing average composition of sam_cols.
    Also plots a single standalone legend for the lipid subtypes once as a separate figure.

    Parameters:
    -----------
    data_df : pd.DataFrame
        The input DataFrame.
    sam_cols : list of str
        The columns representing lipid percentages.
    group_col : str, default 'Source_ID'
        The column to group by.
    unit_col : str, default 'Unit'
        The cell type column.
    units : str, list of str, or None
        Cell type(s) to include. If None, plots one figure for 'All Cells'.
    label_color_dict : dict, default None
        Dictionary mapping column names to styles:
        {'column_name': {'label': 'Custom Label', 'color': 'color_code'}}
        If a column is missing, it will use its default label and be colored gray.
    """
    # 1. Filter out cells where the sum of sam_cols is 0
    df_valid = data_df.copy()
    row_sums = df_valid[sam_cols].sum(axis=1)
    df_valid = df_valid[row_sums > 0]
    
    if df_valid.empty:
        print("Error: No cells found with non-zero lipid composition.")
        return
        
    # 2. Normalize rows to sum to 100%
    row_sums_valid = df_valid[sam_cols].sum(axis=1)
    df_valid[sam_cols] = df_valid[sam_cols].div(row_sums_valid, axis=0) * 100

    # 3. Determine units
    if units is None:
        unit_list = ['All Cells']
    elif isinstance(units, str):
        unit_list = [units]
    else:
        unit_list = list(units)
        
    # 4. Get unique groups
    groups = sorted(df_valid[group_col].unique())
    if len(groups) == 0:
        print("Error: No groups found in group_col.")
        return

    # 5. Handle label and color configurations
    DEFAULT_GRAY = '#D3D3D3'
    processed_labels = []
    processed_colors = []
    legend_elements = []
    
    for col in sam_cols:
        is_mapped = label_color_dict and col in label_color_dict
        if is_mapped:
            label = label_color_dict[col].get('label', col.replace('SAM_', ''))
            color = label_color_dict[col].get('color', DEFAULT_GRAY)
        else:
            label = col.replace('SAM_', '')
            color = DEFAULT_GRAY
            
        processed_labels.append(label)
        processed_colors.append(color)

        # Only add to legend if it is defined in the input dictionary
        if is_mapped:
            legend_elements.append(Patch(facecolor=color, label=label))

    # 6. Plot the single standalone legend once
    n_legend = len(legend_elements)
    if n_legend > 0:
        ncol = n_legend if n_legend <= 5 else int(np.ceil(n_legend / 2.0))
        nrows = 1 if n_legend <= 5 else 2
        
        # Create a compact figure just for the legend
        fig_leg, ax_leg = plt.subplots(figsize=(2 * ncol, 0.4 * nrows))
        ax_leg.axis('off')
        ax_leg.legend(
            handles=legend_elements,
            loc='center',
            ncol=ncol,
            title="Lipid Subtypes",
            fontsize=10,
            title_fontsize=12,
            frameon=True
        )
        plt.tight_layout()
        plt.show()

    # 7. Iterate through each unit and plot as a separate figure (no legends here)
    for unit in unit_list:
        # Filter for the specific unit
        if unit == 'All Cells':
            df_unit = df_valid
        else:
            df_unit = df_valid[df_valid[unit_col] == unit]
            
        if df_unit.empty:
            print(f"Warning: No data found for cell unit '{unit}'")
            continue
            
        # Group by Source_ID and get means
        grouped_means = df_unit.groupby(group_col)[sam_cols].mean()
        
        n_cols = len(groups)
        fig, axes = plt.subplots(1, n_cols, figsize=(4.5 * n_cols, 4.5), squeeze=False)
        
        for col_idx, group in enumerate(groups):
            ax = axes[0, col_idx]
            
            if group not in grouped_means.index:
                ax.text(0.5, 0.5, "No Data", ha='center', va='center')
                ax.axis('off')
                continue
                
            sizes = grouped_means.loc[group]
            
            # Filter out zero-composition slices
            nonzero_idx = sizes > 0.01
            if not nonzero_idx.any():
                ax.text(0.5, 0.5, "No Slices > 0%", ha='center', va='center')
                ax.axis('off')
                continue
                
            group_sizes = sizes[nonzero_idx]
            group_colors = np.array(processed_colors)[nonzero_idx]
            
            # Plot donut chart (no labels on wedges)
            wedges, texts, autotexts = ax.pie(
                group_sizes,
                colors=group_colors,
                autopct='%1.1f%%',
                startangle=90,
                pctdistance=0.75,
                textprops={'fontsize': 9.5},
                wedgeprops={'linewidth': 2.5, 'edgecolor': 'white'}
            )
            
            # Draw center circle to make it a donut chart
            centre_circle = plt.Circle((0,0), 0.55, fc='white')
            ax.add_artist(centre_circle)
            
            # Clean up percentage labels: hide if very small (< 1.5%)
            for autotext, sz in zip(autotexts, group_sizes):
                if sz < 1.5:
                    autotext.set_text('')
                else:
                    autotext.set_fontsize(8.5)
                
            # Set title: column represents group name (without "Group:" prefix)
            ax.set_title(f"{group}", fontsize=13, fontweight='bold')
            ax.axis('equal')
            
            # Set ylabel on the leftmost plot to display the cell unit
            if col_idx == 0:
                ax.set_ylabel(unit, fontsize=12, fontweight='bold', labelpad=20)
            
        plt.tight_layout()
        plt.show()


In [ ]:
# 1. Define custom mapped styles (missing columns will be grayed out and omitted from legend)
lipid_styles = {
    'SAM_Sterols': {'label': 'Sterols', 'color': '#ff9999'},
    'SAM_Cardiolipins': {'label': 'Cardiolipins', 'color': '#66b3ff'},
    'SAM_Phosphatidylcholines': {'label': 'PC', 'color': '#99ff99'},
    'SAM_Phosphatidylethanolamines': {'label': 'PE', 'color': '#ffcc99'},
    'SAM_Ceramides': {'label': 'Ceramides', 'color': '#c2c2f0'},
    'SAM_Glycerols': {'label': 'Glycerols', 'color': '#ffb3e6'},

}

# 2. Plotting for All Cells (as a single row of pie charts)
plot_lipid_composition(
    data_df=data_df, 
    sam_cols=sam_cols, 
    units=None, 
    label_color_dict=lipid_styles
)

In [ ]:
plot_violin_stats(
    data_df=data_df,
    value='Collagen_Coherence',
    compare_col='Unit',
    compare_levels=['FIB', 'MYOF'],
    facet_col='Source_ID',
    summary_type='mean',  # Displays means with .3f precision
    output_dir=output_dir
)

In [ ]:
print(f"\n--- Pairwise Cohen's d ---")
grouped = plot_data_df.groupby(group)[value]
means = grouped.mean()
stds = grouped.std()
counts = grouped.count()

for pair in tukey._results_table[1:]:
    g1, g2 = pair[0].data, pair[1].data
    
    # Calculate Pooled Standard Deviation
    n1, n2 = counts[g1], counts[g2]
    s1, s2 = stds[g1], stds[g2]
    pooled_sd = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    
    # Calculate Cohen's d
    cohens_d = (means[g1] - means[g2]) / pooled_sd
    print(f"{g1} vs {g2}: Cohen's d = {abs(cohens_d):.3f}")

In [ ]:
rf_cols = []
sam_cols = []
for col in srs_df.columns:
    if 'RF' in col:
        rf_cols.append(col)

    elif 'SAM' in col:
        sam_cols.append(col)


In [ ]:
rf_df = srs_df[rf_cols]
rf_df = rf_df.loc[:, ~(rf_df.isin([0, float('NaN')]).all(axis=0))]
sam_df = srs_df[sam_cols]
sam_df = sam_df.loc[:, ~(sam_df.isin([0, float('NaN')]).all(axis=0))]

In [ ]:
import umap

label = rf_df.iloc[:,:3]
features = rf_df.iloc[:, 3:]
features = features.fillna(0)
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
embedding = reducer.fit_transform(features)
df_umap = pd.DataFrame(embedding, columns=['UMAP_1', 'UMAP_2'])
df_umap[label.columns] = rf_df[label.columns].values
s= 5

In [ ]:
import umap

label = sam_df.iloc[:,:3]
features = sam_df.iloc[:, 3:]
features = features.fillna(0)
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
embedding = reducer.fit_transform(features)
df_umap = pd.DataFrame(embedding, columns=['UMAP_1', 'UMAP_2'])
df_umap[label.columns] = sam_df[label.columns].values
s= 5

In [ ]:
sam_df

In [ ]:
import matplotlib.pyplot as plt
hue_col = 'Source_ID'
unique_labels = df_umap[hue_col].unique()
palette  = sns.color_palette("turbo", len(unique_labels))



plt.figure(figsize=(16, 12))
ax = sns.scatterplot(x='UMAP_1', y='UMAP_2', hue=hue_col, s=s, data=df_umap, palette=palette)
plt.title('UMAP Projection of Sample Source', fontsize=20, fontweight='bold')
plt.xlabel('UMAP 1', fontsize=16)
plt.ylabel('UMAP 2', fontsize=16)
plt.legend(markerscale=3.0, fontsize=12)
# plt.savefig(os.path.join(output_dir,f"UMAP_Source.png"),dpi=300, transparent=False)
plt.show()

In [ ]:

import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statannotations.Annotator import Annotator
import statsmodels.api as sm


display_names = {
    "16:0 Cardiolipin": "CL 18:2",
    "PC Mix": "PC",
    "PE Mix": "PE",
    "PS Mix": "PS",
    "PI Mix": "PI",
    "PG Mix": "PG",
    "18:0 Lyso PA": "LPA 18:0",
    "18:1 LPA": "LPA 18:1",
    "DOPC": "PC(18:1/18:1)",
    "DOPE": "PE(18:1/18:1)",
    "DSPC": "PC(18:0/18:0)",
    "16:0 CDP DG": "CDP-DG 16:0",
    "Cholesterol (ovine)": "Chol",
    "18:1 Cholesterol Ester": "Chol Ester 18:1",
    "TAG Mix": "TAG",
    "TAG 16:0": "TAG 16:0",
    "TAG 18:1": "TAG 18:1",
    "DAG 16:0": "DAG 16:0",
    "DAG 18:0 and 24:0 ": "DAG(18:0/24:0)",
    "Sphingosine": "SM",
    "Cer 18:1-12:0": "Cer(d18:1/12:0)",
    "Cer 18:1-18:0": "Cer(d18:1/18:0)",
    "Cer 18:1-18:1": "Cer(d18:1/18:1)",
    "Cer 18:1-22:0": "Cer(d18:1/22:0)",
    "Cer 18:1-24:0": "Cer(d18:1/24:0)",
    "Cer 18:1-24:1": "Cer(d18:1/24:1)",
    "Cer 18:0-14:0": "dhCer(d18:0/14:0)",
    "Cer 18:1-24:1(15Z)": "dhCer(d18:0/24:1)",
    "Cer m18:1-16:0": "doxCer(m18:1/16:0)",
    "Cer m18:1-24:1": "doxCer(m18:1/24:1)",
    "Glucosylceramide (Gaucher's Spleen)": "GlcCer",
    "C8 L-Carnitine": "CAR 8:0",
    "C12 Carnitine": "CAR 12:0",
    "C16 Carnitine": "CAR 16:0",
    "C18 Carnitine (d9-cis)": "CAR 18:0-d9",
    "Stearic Acid": "FA 18:0",
    "Palmitic Acid": "FA 16:0",
    "Docosahexaenoic Acid": "FA 22:6",
    "Tetracosapentaenoic Acid": "FA 24:5",
}


hue_col = 'Source_ID'
unique_labels = df_umap[hue_col].unique()
palette  = sns.color_palette("turbo", len(unique_labels))


group = 'Source_ID'


for mol in rf_cols:
    mol_name = mol.replace('RF_', '')
    display_mol = display_names[mol_name]
    value = mol

    for unit in rf_df['Unit'].unique():
        try:
            subset_df = rf_df[rf_df['Unit']==unit]
            plot_data_df = subset_df.dropna(subset=[value])

            df_counts = plot_data_df[group].value_counts()
            valid_mask = df_counts[df_counts >= 2].index 
            plot_data_df = plot_data_df[plot_data_df[group].isin(valid_mask)]

            # groups = [rf_df[rf_df['Source_ID'] == source][value] for source in rf_df['Source_ID'].unique()]
            result = sm.stats.anova_oneway(plot_data_df[value], groups=plot_data_df[group], use_var='equal')
            exact_p_value = 1 - stats.f.cdf(result.statistic, result.df_num, result.df_denom)

            tukey = pairwise_tukeyhsd(endog=plot_data_df[value], groups=plot_data_df[group], alpha=0.05)
            tukey_summary = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
            pvalues = tukey.pvalues

            group1_col = tukey_summary.iloc[:, 0]
            group2_col = tukey_summary.iloc[:, 1]


            pairs= [(g1, g2) for g1, g2 in zip(group1_col, group2_col)]

            plt.figure(figsize=(8, 6))
            ax = sns.violinplot(data=plot_data_df, x=group, y=value, palette=palette)

            ax.set_axisbelow(True)
            ax.grid(True, alpha=0.7, linestyle='--')

            annotator = Annotator(ax, pairs, data=plot_data_df, x=group, y=value)

            annotator.configure(text_format="star", loc="outside")
            annotator.set_pvalues(pvalues)

            annotator.annotate()
            sns.despine()
            ax.set_ylabel('Rel. Pct. (%)', fontsize=12)
            ax.set_xlabel("", fontsize=12)
            plt.suptitle(f"{display_mol}: {unit}", fontsize=16, fontweight='bold', y=0.9 if len(pairs) > 1 else 0.95 , x = 0.53, ha='center')
            plt.title(f"ANOVA p-value: {exact_p_value:.2e}", y=1.3 if len(pairs) > 1 else 1.1 , fontsize=12, color='gray', pad=10)
            plt.tight_layout()
            mol_safe = display_mol.replace("/","-").replace(":","_").replace(" ","_")
            vp_output_dir = os.path.join(output_dir, f"rf-{mol_safe}")
            os.makedirs(vp_output_dir, exist_ok=True)
            plt.savefig(os.path.join(vp_output_dir,f'vplot_{unit.replace("/","")}.png'),dpi=300, transparent=False)
            plt.savefig(os.path.join(vp_output_dir,f'vplot_{unit.replace("/","")}_trans.png'),dpi=300, transparent=True)
            # plt.show()
            plt.close()
        except Exception as e: print(f"    Violin plot failed for {unit}: {e}")

In [ ]:

import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statannotations.Annotator import Annotator
import statsmodels.api as sm


display_names = {
    "16:0 Cardiolipin": "CL 18:2",
    "PC Mix": "PC",
    "PE Mix": "PE",
    "PS Mix": "PS",
    "PI Mix": "PI",
    "PG Mix": "PG",
    "18:0 Lyso PA": "LPA 18:0",
    "18:1 LPA": "LPA 18:1",
    "DOPC": "PC(18:1/18:1)",
    "DOPE": "PE(18:1/18:1)",
    "DSPC": "PC(18:0/18:0)",
    "16:0 CDP DG": "CDP-DG 16:0",
    "Cholesterol (ovine)": "Chol",
    "18:1 Cholesterol Ester": "Chol Ester 18:1",
    "TAG Mix": "TAG",
    "TAG 16:0": "TAG 16:0",
    "TAG 18:1": "TAG 18:1",
    "DAG 16:0": "DAG 16:0",
    "DAG 18:0 and 24:0 ": "DAG(18:0/24:0)",
    "Sphingosine": "SM",
    "Cer 18:1-12:0": "Cer(d18:1/12:0)",
    "Cer 18:1-18:0": "Cer(d18:1/18:0)",
    "Cer 18:1-18:1": "Cer(d18:1/18:1)",
    "Cer 18:1-22:0": "Cer(d18:1/22:0)",
    "Cer 18:1-24:0": "Cer(d18:1/24:0)",
    "Cer 18:1-24:1": "Cer(d18:1/24:1)",
    "Cer 18:0-14:0": "dhCer(d18:0/14:0)",
    "Cer 18:1-24:1(15Z)": "dhCer(d18:0/24:1)",
    "Cer m18:1-16:0": "doxCer(m18:1/16:0)",
    "Cer m18:1-24:1": "doxCer(m18:1/24:1)",
    "Glucosylceramide (Gaucher's Spleen)": "GlcCer",
    "C8 L-Carnitine": "CAR 8:0",
    "C12 Carnitine": "CAR 12:0",
    "C16 Carnitine": "CAR 16:0",
    "C18 Carnitine (d9-cis)": "CAR 18:0-d9",
    "Stearic Acid": "FA 18:0",
    "Palmitic Acid": "FA 16:0",
    "Docosahexaenoic Acid": "FA 22:6",
    "Tetracosapentaenoic Acid": "FA 24:5",
}


hue_col = 'Source_ID'
unique_labels = df_umap[hue_col].unique()
palette  = sns.color_palette("turbo", len(unique_labels))


group = 'Source_ID'


for mol in sam_cols:
    mol_name = mol.replace('RF_', '')
    display_mol = display_names[mol_name]
    value = mol

    for unit in sam_df['Unit'].unique():
        try:
            subset_df = sam_df[sam_df['Unit']==unit]
            plot_data_df = subset_df.dropna(subset=[value])

            df_counts = plot_data_df[group].value_counts()
            valid_mask = df_counts[df_counts >= 2].index 
            plot_data_df = plot_data_df[plot_data_df[group].isin(valid_mask)]

            # groups = [sam_df[sam_df['Source_ID'] == source][value] for source in sam_df['Source_ID'].unique()]
            result = sm.stats.anova_oneway(plot_data_df[value], groups=plot_data_df[group], use_var='equal')
            exact_p_value = 1 - stats.f.cdf(result.statistic, result.df_num, result.df_denom)

            tukey = pairwise_tukeyhsd(endog=plot_data_df[value], groups=plot_data_df[group], alpha=0.05)
            tukey_summary = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
            pvalues = tukey.pvalues

            group1_col = tukey_summary.iloc[:, 0]
            group2_col = tukey_summary.iloc[:, 1]


            pairs= [(g1, g2) for g1, g2 in zip(group1_col, group2_col)]

            plt.figure(figsize=(8, 6))
            ax = sns.violinplot(data=plot_data_df, x=group, y=value, palette=palette)

            ax.set_axisbelow(True)
            ax.grid(True, alpha=0.7, linestyle='--')

            annotator = Annotator(ax, pairs, data=plot_data_df, x=group, y=value)

            annotator.configure(text_format="star", loc="outside")
            annotator.set_pvalues(pvalues)

            annotator.annotate()
            sns.despine()
            ax.set_ylabel('Rel. Pct. (%)', fontsize=12)
            ax.set_xlabel("", fontsize=12)
            plt.suptitle(f"{display_mol}: {unit}", fontsize=16, fontweight='bold', y=0.9 if len(pairs) > 1 else 0.95 , x = 0.53, ha='center')
            plt.title(f"ANOVA p-value: {exact_p_value:.2e}", y=1.3 if len(pairs) > 1 else 1.1 , fontsize=12, color='gray', pad=10)
            plt.tight_layout()
            mol_safe = display_mol.replace("/","-").replace(":","_").replace(" ","_")
            vp_output_dir = os.path.join(output_dir, f"sam-{mol_safe}")
            os.makedirs(vp_output_dir, exist_ok=True)
            plt.savefig(os.path.join(vp_output_dir,f'vplot_{unit.replace("/","")}.png'),dpi=300, transparent=False)
            # plt.savefig(os.path.join(vp_output_dir,f'vplot_{unit.replace("/","")}_trans.png'),dpi=300, transparent=True)
            # plt.show()
            plt.close()
        except Exception as e: print(f"    Violin plot failed for {unit}: {e}")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats import multitest

def create_unit_group_bubble_chart(df, value_col, unit_col='Unit', value_name=None, unit_order=None, group_col='Source_ID',
                                    output_dir=None, alpha=0.05, min_group_n=2,
                                    test='mannwhitney', correction='fdr_bh',
                                    zscore_axis='groups', size_scale=220, size_cap=700,
                                    hide_non_significant=True, show_values=False,
                                    cmap='RdBu_r', figsize=None, grid_spacing=1.9, padding=0.9, min_marker_size=65,
                                    save_name=None, show_plot=True,
                                    orient='v',
                                    save_cbar_separate=False):
    """
    Produce a bubble chart: 
    If orient='v' (default): y=units, x=groups
    If orient='h': y=groups (top to bottom), x=units
    color= z-scored mean (by unit or group),
    size = -log10(adj_p) from one-vs-all tests (default Mann-Whitney U, BH-corrected).
    
    If save_cbar_separate=True, the colorbar is saved as a separate file and omitted from the main plot.
    Returns (fig, ax, diagnostics_df, save_path).
    """
    df = df.copy()
    df = df.dropna(subset=[value_col, unit_col, group_col]).reset_index(drop=True)

    # Prepare output folder
    if output_dir is None:
        output_dir = os.path.join(os.getcwd(), 'bubble_charts')
    os.makedirs(output_dir, exist_ok=True)

    # Aggregate means and counts
    means = df.groupby([unit_col, group_col])[value_col].mean().unstack(fill_value=np.nan)
    counts = df.groupby([unit_col, group_col])[value_col].count().unstack(fill_value=0)

    units = list(means.index.astype(str))
    groups = list(means.columns.astype(str))

    records = []
    # collect p-values for each unit x group (one-vs-all for that unit)
    for unit in units:
        for group in groups:
            n_in = int(counts.loc[unit, group]) if group in counts.columns and unit in counts.index else 0
            mean_val = float(means.loc[unit, group]) if group in means.columns and unit in means.index and not np.isnan(means.loc[unit, group]) else np.nan

            if n_in < min_group_n:
                records.append({'unit': unit, 'group': group, 'mean': mean_val, 'n_in': n_in, 'n_out': np.nan, 'raw_p': np.nan})
                continue

            # samples
            sample_in = df[(df[unit_col].astype(str) == unit) & (df[group_col].astype(str) == group)][value_col].dropna()
            sample_out = df[(df[unit_col].astype(str) == unit) & (df[group_col].astype(str) != group)][value_col].dropna()
            n_out = len(sample_out)
            if len(sample_out) < min_group_n:
                records.append({'unit': unit, 'group': group, 'mean': mean_val, 'n_in': n_in, 'n_out': n_out, 'raw_p': np.nan})
                continue

            # perform test
            try:
                if test == 'mannwhitney':
                    stat, p = mannwhitneyu(sample_in, sample_out, alternative='two-sided')
                elif test == 'ttest':
                    from scipy.stats import ttest_ind
                    stat, p = ttest_ind(sample_in, sample_out, equal_var=False, nan_policy='omit')
                else:
                    stat, p = mannwhitneyu(sample_in, sample_out, alternative='two-sided')
            except Exception:
                p = np.nan

            records.append({'unit': unit, 'group': group, 'mean': mean_val, 'n_in': n_in, 'n_out': n_out, 'raw_p': p})

    diag = pd.DataFrame.from_records(records)
    # multiple testing correction
    mask_tests = diag['raw_p'].notna()
    if mask_tests.any():
        corrected = multitest.multipletests(diag.loc[mask_tests, 'raw_p'].values, alpha=alpha, method=( 'fdr_bh' if correction=='fdr_bh' else 'bonferroni' ))
        adj_p = np.full(len(diag), np.nan)
        adj_p[mask_tests.values] = corrected[1]
        diag['adj_p'] = adj_p
    else:
        diag['adj_p'] = np.nan

    # size mapping: -log10(adj_p) layered on top of a minimum visible marker size
    p_floor = 1e-300
    adj = diag['adj_p'].fillna(1.0).clip(lower=p_floor)
    sizes = min_marker_size + (np.log1p(-np.log10(adj)) * size_scale)
    if size_cap is not None:
        sizes = sizes.clip(upper=size_cap)
    diag['size'] = sizes
    if hide_non_significant:
        diag.loc[diag['adj_p'] >= alpha, 'size'] = min_marker_size

    # compute z-score of means
    mean_pivot = means.copy()
    if zscore_axis == 'rows':
        z = mean_pivot.sub(mean_pivot.mean(axis=1), axis=0).div(mean_pivot.std(axis=1).replace(0, np.nan), axis=0)
    elif zscore_axis in ('groups', 'columns'):
        z = mean_pivot.sub(mean_pivot.mean(axis=0), axis=1).div(mean_pivot.std(axis=0).replace(0, np.nan), axis=1)
    else:
        raise ValueError('zscore_axis must be one of: "rows", "groups"/"columns"')
        
    # map z-scores back to diag
    zs = []
    for _, row in diag.iterrows():
        try:
            zs.append(z.loc[row['unit'], row['group']])
        except Exception:
            zs.append(np.nan)
    diag['zscore'] = zs

    # plotting preparation
    units_order = units if unit_order is None else unit_order
    groups_order = ['REF', 'AKI', 'CKD']
    
    # Map coordinates based on orientation
    if orient == 'h':
        x_map = {u: i * grid_spacing for i, u in enumerate(units_order)}
        y_map = {g: i * grid_spacing for i, g in enumerate(groups_order)}
        diag['x'] = diag['unit'].map(x_map)
        diag['y'] = diag['group'].map(y_map)
    else:
        x_map = {g: i * grid_spacing for i, g in enumerate(groups_order)}
        y_map = {u: i * grid_spacing for i, u in enumerate(units_order)}
        diag['x'] = diag['group'].map(x_map)
        diag['y'] = diag['unit'].map(y_map)

    # create scatter arrays
    plot_df = diag.copy()
    draw_mask = plot_df['size'] > 0
    x_vals = plot_df.loc[draw_mask, 'x'].values
    y_vals = plot_df.loc[draw_mask, 'y'].values
    s_vals = plot_df.loc[draw_mask, 'size'].values
    c_vals = plot_df.loc[draw_mask, 'zscore'].values
    
    # Adjust default figsize for horizontal layouts
    if figsize is None:
        if orient == 'h':
            figsize = (max(8, len(units_order) * 0.9), max(4, len(groups_order) * 1.3))
        else:
            figsize = (max(6, len(groups_order) * 2.0), max(8, len(units_order) * 0.45))
            
    fig, ax = plt.subplots(figsize=figsize)
    if len(x_vals):
        sc = ax.scatter(x_vals, y_vals, s=s_vals, c=c_vals, vmin=-2.5, vmax=2.5, cmap=cmap, edgecolor='k', linewidth=0.3)
        
        # Colorbar handling
        if save_cbar_separate:
            fig_cbar, ax_cbar = plt.subplots(figsize=(1.2, 4.0))
            cb = fig_cbar.colorbar(sc, cax=ax_cbar)
            cb.set_label('Z-score', fontsize=18, fontweight='bold')
            cbar_save_name = f'cbar_{value_name}.png'
            cbar_save_path = os.path.join(output_dir, cbar_save_name)
            fig_cbar.savefig(cbar_save_path, dpi=300, bbox_inches='tight', transparent=True)
            plt.close(fig_cbar)
            print(f"Colorbar saved separately in {output_dir} as {cbar_save_name}")
        else:
            cb = plt.colorbar(sc, ax=ax)
            cb.set_label('Z-score of mean')
    else:
        sc = None

    # Set axis ticks and labels based on orientation
    if orient == 'h':
        ax.set_xticks([x_map[u] for u in units_order])
        ax.set_xticklabels(units_order, fontweight='bold', fontsize=14, rotation=45, ha='right')
        ax.set_yticks([y_map[g] for g in groups_order])
        ax.set_yticklabels(groups_order, fontweight='bold', fontsize=14)
        ax.set_xlim(-padding, (len(units_order) - 1) * grid_spacing + padding)
        ax.set_ylim(-padding, (len(groups_order) - 1) * grid_spacing + padding)
        ax.invert_yaxis() # Invert Y-axis so group order goes top to bottom: REF -> AKI -> CKD
    else:
        ax.set_xticks([x_map[g] for g in groups_order])
        ax.set_xticklabels(groups_order, fontweight='bold', fontsize=14)
        ax.set_yticks([y_map[u] for u in units_order])
        ax.set_yticklabels(units_order, fontweight='bold', fontsize=14)
        ax.set_xlim(-padding, (len(groups_order) - 1) * grid_spacing + padding)
        ax.set_ylim(-padding, (len(units_order) - 1) * grid_spacing + padding)
        
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(f'{value_name.replace("_"," ")}', fontsize=14, fontweight='bold')
    ax.set_aspect('equal', adjustable='box')
    plt.tight_layout()
    plt.show()

    # optionally annotate means
    if show_values and len(x_vals):
        for _, r in plot_df.loc[draw_mask].iterrows():
            ax.text(r['x'], r['y'], f"{r['mean']:.2f}", fontsize=7, va='center', ha='center')

    # save figure
    save_suffix = f'_{orient}' if orient == 'h' else ''
    save_name = save_name or f'bubble_{value_name}{save_suffix}.png'
    save_path = os.path.join(output_dir, save_name)

    fig.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)

    if not show_plot:
        plt.close(fig)

    return fig, ax, diag, save_path


In [ ]:
def create_unit_group_bubble_chart_raw(df, value_col, unit_col='Unit', value_name=None, unit_order=None, group_col='Source_ID',
                                        output_dir=None, alpha=0.05, min_group_n=2,
                                        test='mannwhitney', correction='fdr_bh',
                                        size_scale=220, size_cap=700,
                                        hide_non_significant=True, show_values=False,
                                        cmap='RdBu_r', figsize=None, grid_spacing=1.9, padding=0.9, min_marker_size=65,
                                        save_name=None, show_plot=True,
                                        orient='v',
                                        save_cbar_separate=False,
                                        vmin=None, vmax=None):
    """
    Produce a bubble chart: 
    If orient='v' (default): y=units, x=groups
    If orient='h': y=groups (top to bottom), x=units
    color= raw mean value (by unit and group),
    size = -log10(adj_p) from one-vs-all tests (default Mann-Whitney U, BH-corrected).
    
    If save_cbar_separate=True, the colorbar is saved as a separate file and omitted from the main plot.
    Returns (fig, ax, diagnostics_df, save_path).
    """
    df = df.copy()
    df = df.dropna(subset=[value_col, unit_col, group_col]).reset_index(drop=True)

    # Prepare output folder
    if output_dir is None:
        output_dir = os.path.join(os.getcwd(), 'bubble_charts')
    os.makedirs(output_dir, exist_ok=True)

    # Aggregate means and counts
    means = df.groupby([unit_col, group_col])[value_col].mean().unstack(fill_value=np.nan)
    counts = df.groupby([unit_col, group_col])[value_col].count().unstack(fill_value=0)

    units = list(means.index.astype(str))
    groups = list(means.columns.astype(str))

    records = []
    # collect p-values for each unit x group (one-vs-all for that unit)
    for unit in units:
        for group in groups:
            n_in = int(counts.loc[unit, group]) if group in counts.columns and unit in counts.index else 0
            mean_val = float(means.loc[unit, group]) if group in means.columns and unit in means.index and not np.isnan(means.loc[unit, group]) else np.nan

            if n_in < min_group_n:
                records.append({'unit': unit, 'group': group, 'mean': mean_val, 'n_in': n_in, 'n_out': np.nan, 'raw_p': np.nan})
                continue

            # samples
            sample_in = df[(df[unit_col].astype(str) == unit) & (df[group_col].astype(str) == group)][value_col].dropna()
            sample_out = df[(df[unit_col].astype(str) == unit) & (df[group_col].astype(str) != group)][value_col].dropna()
            n_out = len(sample_out)
            if len(sample_out) < min_group_n:
                records.append({'unit': unit, 'group': group, 'mean': mean_val, 'n_in': n_in, 'n_out': n_out, 'raw_p': np.nan})
                continue

            # perform test
            try:
                if test == 'mannwhitney':
                    stat, p = mannwhitneyu(sample_in, sample_out, alternative='two-sided')
                elif test == 'ttest':
                    from scipy.stats import ttest_ind
                    stat, p = ttest_ind(sample_in, sample_out, equal_var=False, nan_policy='omit')
                else:
                    stat, p = mannwhitneyu(sample_in, sample_out, alternative='two-sided')
            except Exception:
                p = np.nan

            records.append({'unit': unit, 'group': group, 'mean': mean_val, 'n_in': n_in, 'n_out': n_out, 'raw_p': p})

    diag = pd.DataFrame.from_records(records)
    # multiple testing correction
    mask_tests = diag['raw_p'].notna()
    if mask_tests.any():
        corrected = multitest.multipletests(diag.loc[mask_tests, 'raw_p'].values, alpha=alpha, method=( 'fdr_bh' if correction=='fdr_bh' else 'bonferroni' ))
        adj_p = np.full(len(diag), np.nan)
        adj_p[mask_tests.values] = corrected[1]
        diag['adj_p'] = adj_p
    else:
        diag['adj_p'] = np.nan

    # size mapping: -log10(adj_p) layered on top of a minimum visible marker size
    p_floor = 1e-300
    adj = diag['adj_p'].fillna(1.0).clip(lower=p_floor)
    sizes = min_marker_size + (np.log1p(-np.log10(adj)) * size_scale)
    if size_cap is not None:
        sizes = sizes.clip(upper=size_cap)
    diag['size'] = sizes
    if hide_non_significant:
        diag.loc[diag['adj_p'] >= alpha, 'size'] = min_marker_size

    # plotting preparation
    units_order = units if unit_order is None else unit_order
    groups_order = ['REF', 'AKI', 'CKD']
    
    # Map coordinates based on orientation
    if orient == 'h':
        x_map = {u: i * grid_spacing for i, u in enumerate(units_order)}
        y_map = {g: i * grid_spacing for i, g in enumerate(groups_order)}
        diag['x'] = diag['unit'].map(x_map)
        diag['y'] = diag['group'].map(y_map)
    else:
        x_map = {g: i * grid_spacing for i, g in enumerate(groups_order)}
        y_map = {u: i * grid_spacing for i, u in enumerate(units_order)}
        diag['x'] = diag['group'].map(x_map)
        diag['y'] = diag['unit'].map(y_map)

    # create scatter arrays
    plot_df = diag.copy()
    draw_mask = plot_df['size'] > 0
    x_vals = plot_df.loc[draw_mask, 'x'].values
    y_vals = plot_df.loc[draw_mask, 'y'].values
    s_vals = plot_df.loc[draw_mask, 'size'].values
    c_vals = plot_df.loc[draw_mask, 'mean'].values
    
    # Adjust default figsize for horizontal layouts
    if figsize is None:
        if orient == 'h':
            figsize = (max(8, len(units_order) * 0.9), max(4, len(groups_order) * 1.3))
        else:
            figsize = (max(6, len(groups_order) * 2.0), max(8, len(units_order) * 0.45))
            
    fig, ax = plt.subplots(figsize=figsize)
    if len(x_vals):
        sc = ax.scatter(x_vals, y_vals, s=s_vals, c=c_vals, vmin=vmin, vmax=vmax, cmap=cmap, edgecolor='k', linewidth=0.3)
        
        # Colorbar handling
        if save_cbar_separate:
            fig_cbar, ax_cbar = plt.subplots(figsize=(1.2, 4.0))
            cb = fig_cbar.colorbar(sc, cax=ax_cbar)
            cb.set_label('Mean value', fontsize=12, fontweight='bold', rotation=90)
            cbar_save_name = f'cbar_{value_name}.png'
            cbar_save_path = os.path.join(output_dir, cbar_save_name)
            fig_cbar.savefig(cbar_save_path, dpi=300, bbox_inches='tight', transparent=True)
            plt.close(fig_cbar)
            print(f"Colorbar saved separately in {output_dir} as {cbar_save_name}")
        else:
            cb = plt.colorbar(sc, ax=ax)
            cb.set_label('Mean value')
    else:
        sc = None

    # Set axis ticks and labels based on orientation
    if orient == 'h':
        ax.set_xticks([x_map[u] for u in units_order])
        ax.set_xticklabels(units_order, fontweight='bold', fontsize=14, rotation=45, ha='right', rotation_mode='anchor' )
        ax.set_yticks([y_map[g] for g in groups_order])
        ax.set_yticklabels(groups_order, fontweight='bold', fontsize=14)
        ax.set_xlim(-padding, (len(units_order) - 1) * grid_spacing + padding)
        ax.set_ylim(-padding, (len(groups_order) - 1) * grid_spacing + padding)
        ax.invert_yaxis() # Invert Y-axis so group order goes top to bottom: REF -> AKI -> CKD
    else:
        ax.set_xticks([x_map[g] for g in groups_order])
        ax.set_xticklabels(groups_order, fontweight='bold', fontsize=14)
        ax.set_yticks([y_map[u] for u in units_order])
        ax.set_yticklabels(units_order, fontweight='bold', fontsize=14)
        ax.set_xlim(-padding, (len(groups_order) - 1) * grid_spacing + padding)
        ax.set_ylim(-padding, (len(units_order) - 1) * grid_spacing + padding)
        
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(f'{value_name.replace("_"," ")}', fontsize=14, fontweight='bold')
    ax.set_aspect('equal', adjustable='box')
    plt.tight_layout()
    plt.show()

    # optionally annotate means
    if show_values and len(x_vals):
        for _, r in plot_df.loc[draw_mask].iterrows():
            ax.text(r['x'], r['y'], f"{r['mean']:.2f}", fontsize=7, va='center', ha='center')

    # save figure
    save_suffix = f'_{orient}' if orient == 'h' else ''
    save_name = save_name or f'bubble_{value_name}{save_suffix}.png'
    save_path = os.path.join(output_dir, save_name)

    fig.savefig(save_path, dpi=300, bbox_inches='tight', transparent=True)

    if not show_plot:
        plt.close(fig)

    return fig, ax, diag, save_path


In [ ]:
data_df = srs_df[data_cols]

display_val = {
    'SAM_Sterols' : 'Sterols'

}

value = 'SAM_Glycerols'
zscore_axis = 'columns'
fig, ax, diag_df, path = create_unit_group_bubble_chart_raw(
    data_df, value, value_name=display_val.get(value, value), unit_order=unit_order[::-1], output_dir=output_dir,  figsize=(5,5),
    alpha=0.05, min_group_n=2, show_plot=False, show_values=False, save_name=f'{display_val.get(value, value)}-dyslip_bubble_chart.png', orient='v', save_cbar_separate=True
)

In [ ]:
sam_cols

In [ ]:
unit_order = [
    'POD',
    # 'PEC',
        # 'REN', 
    # 'MD',
    'PT', 
    'altPT', 
    # 'DTL', 
    # 'ATL', 
    'TAL', 
    'altTAL', 
    'DCT', 
        'altDCT', 
    'CNT', 
        'altCNT', 
    # 'PC', 
    # 'IC', 
    # 'altPOD', 

    # 'altDTL', 
    # 'altATL', 

    # 'EC',

    # 'altPC', 
    # 'altIC', 
    # 'FIB', 
    # 'aFIB', 
    # 'pvFIB', 
    # 'infFIB', 
    # 'MYOF',

    # 'EC-GC',

    # 'Lymphoid',
        # 'T',
            # 'Myeloid',
                # 'MON',
    # 'DC',
    # 'B',

    # 'N',
    # 'NK',
    # 'PL',

    # 'MAC',
    # 'resMAC',
    # 'moMAC',
    # 'moMAC-INF',
    # 'MAST',
    # 'NEU',
    # 'SC/NEU',
    # 'MC',

    # 'VSM/P',
    # 'unassigned',
    # 'low_quality',
    ]

In [ ]:
def keep_group(g):
    # Count how many values > 0 exist for each col2 subgroup
    subgroup_counts = (g[sam_cols] > 0).groupby(g['Source_ID']).sum()
    
    # Check if BOTH columns in ALL subgroups have a count strictly greater than 2
    return (subgroup_counts > 2).all().all()

# Apply the filter
filtered_df = data_df.groupby('Unit').filter(keep_group)
print(filtered_df)

In [ ]:
filtered_df[sam_cols]

In [ ]:
df_bool = data_df[sam_cols] > 1e-6
a = df_bool.groupby([data_df['Unit'], data_df['Source_ID'],]).sum().reset_index()

In [ ]:
data_cols.extend(sam_cols)

In [ ]:
filtered_df['Unit']

In [ ]:
data_cols